In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from pprint import pformat

from hloc import (
    extract_features,
    match_features,
    match_dense,
    pairs_from_covisibility,
    pairs_from_retrieval,
)
from hloc import colmap_from_nvm, triangulation, localize_sfm, visualization
import numpy as np
import pandas as pd
from pipeline_utils import read_images_text, colmap_dict_to_pose
import matplotlib.pyplot as plt

# Pipeline for UCF 3259 visual localization

In [2]:
dataset = Path("/home/shubham/Downloads/unknot_codebase/visual_slam/Skydio_alignment_mesh/mesh_outputs")  # change this if your dataset is somewhere else
images = dataset / "0"
outputs = dataset / "temp_outputs"  # where everything will be saved

# list the standard configurations available
print(f"Configs for feature extractors:\n{pformat(extract_features.confs)}")
print(f"Configs for feature matchers:\n{pformat(match_features.confs)}")
print(f"Configs for feature matchers:\n{pformat(match_dense.confs)}")

Configs for feature extractors:
{'aliked-n16': {'model': {'model_name': 'aliked-n16', 'name': 'aliked'},
                'output': 'feats-aliked-n16',
                'preprocessing': {'grayscale': False, 'resize_max': 1024}},
 'd2net-ss': {'model': {'multiscale': False, 'name': 'd2net'},
              'output': 'feats-d2net-ss',
              'preprocessing': {'grayscale': False, 'resize_max': 1600}},
 'dir': {'model': {'name': 'dir'},
         'output': 'global-feats-dir',
         'preprocessing': {'resize_max': 1024}},
 'disk': {'model': {'max_keypoints': 5000, 'name': 'disk'},
          'output': 'feats-disk',
          'preprocessing': {'grayscale': False, 'resize_max': 1600}},
 'disk2': {'model': {'max_keypoints': 4000, 'name': 'disk'},
           'output': 'feats-disk2',
           'preprocessing': {'grayscale': False, 'resize_max': 720}},
 'eigenplaces': {'model': {'name': 'eigenplaces'},
                 'output': 'global-feats-eigenplaces',
                 'preprocessing': 

In [8]:
# pick one of the configurations for image retrieval, local feature extraction, and matching
retrieval_conf = extract_features.confs["netvlad"]

feature_conf = extract_features.confs["superpoint_aachen"]
matcher_conf = match_features.confs["superglue"]
# matcher_conf = match_dense.confs['loftr']

reference_sfm = outputs / "sfm_sift+NN"  # the SfM model we will build

## Extract local features for database

In [9]:
# NOT FOR LOFTR
if matcher_conf["model"]["name"] != "loftr":
    features = extract_features.main(feature_conf, images, outputs)

[2025/06/03 12:41:30 hloc INFO] Extracting local features with configuration:
{'model': {'name': 'dir'},
 'output': 'global-feats-dir',
 'preprocessing': {'resize_max': 1024}}
[2025/06/03 12:41:30 hloc INFO] Found 2 images in root /home/shubham/Downloads/unknot_codebase/visual_slam/Skydio_alignment_mesh/mesh_outputs/0.
/home/shubham/Downloads/unknot_codebase/Hierarchical-Localization/hloc/extractors/../../third_party/deep-image-retrieval/dirtorch/utils/common.py:121: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no lon

=> loading checkpoint '/home/shubham/.cache/torch/hub/dirtorch/Resnet-101-AP-GeM.pt' (current_iter 296)


100%|█████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  4.22it/s]
[2025/06/03 12:41:32 hloc INFO] Finished exporting features.


The function returns the path of the file in which all the extracted features are stored.

## Generate pairs for the SfM reconstruction
Instead of matching all database images exhaustively, we exploit the existing SIFT model to find which image pairs are the most covisible. We do a covisiblity search, selecting the top 20 most covisibile neighbors for each image.

In [10]:
sfm_pairs = outputs / "pairs.txt"

## Match the database images

In [11]:
if matcher_conf["model"]["name"] != "loftr":
    sfm_matches = match_features.main(
        matcher_conf, sfm_pairs, feature_conf["output"], outputs
    )

[2025/06/03 12:42:00 hloc INFO] Matching local features with configuration:
{'model': {'name': 'superglue',
           'sinkhorn_iterations': 50,
           'weights': 'outdoor'},
 'output': 'matches-superglue'}


Loaded SuperGlue model ("outdoor" weights)


/home/shubham/Downloads/unknot_codebase/Hierarchical-Localization/hloc/matchers/../../third_party/SuperGluePretrainedNetwork/models/superglue.py:226: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this e

matcher image size: (np.int64(3456), np.int64(4608))


  0%|                                                                                                 | 0/1 [00:00<?, ?it/s]


AssertionError: Missing key keypoints0 in data

In [7]:
# LOFTR
if matcher_conf["model"]["name"] == "loftr":
    features, sfm_matches = match_dense.main(
            matcher_conf, sfm_pairs, images, outputs, max_kps=8192, overwrite=False
        )

[2025/06/03 12:24:14 hloc INFO] Extracting semi-dense features with configuration:
{'cell_size': 1,
 'max_error': 1,
 'model': {'name': 'loftr', 'weights': 'outdoor'},
 'output': 'matches-loftr',
 'preprocessing': {'dfactor': 8, 'grayscale': True, 'resize_max': 1024}}
[2025/06/03 12:24:15 hloc INFO] Performing dense matching...
  0%|                                                                                                 | 0/1 [00:00<?, ?it/s]

Image shapes:  torch.Size([1, 1, 768, 1024]) torch.Size([1, 1, 768, 1024])


100%|█████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.09s/it]
[2025/06/03 12:24:16 hloc INFO] Assigning matches...
[2025/06/03 12:24:16 hloc INFO] Aggregating keypoints for 2 images.
100%|█████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 70.14it/s]
[2025/06/03 12:24:16 hloc INFO] Finished assignment, found 404.0 keypoints/image (avg.), total 808.
[2025/06/03 12:24:16 hloc INFO] Reassign matches with max_error=1.
100%|████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 167.00it/s]


## Triangulate a new SfM model from the given poses
We triangulate the sparse 3D pointcloud given the matches and the reference poses stored in the SIFT COLMAP model.

In [ ]:
reconstruction = triangulation.main(
    reference_sfm, outputs / "sfm_sift", images, sfm_pairs, features, sfm_matches
)

## Find image pairs via image retrieval
We extract global descriptors with NetVLAD and find for each image the $k$ most similar ones. A larger $k$ improves the robustness of the localization for difficult queries but makes the matching more expensive. Using $k{=}10{-}20$ is generally a good tradeoff.

In [ ]:
global_descriptors = extract_features.main(retrieval_conf, images, outputs)

# Query Images

In [ ]:
num_retrival=20
retrival = "netvlad"
query_set = 0
retrieval_conf = extract_features.confs[retrival]

query_images = dataset / f"query_{query_set}"
outputs_query = outputs / f"query_{query_set}"
loc_pairs = outputs / f"pairs-query_{query_set}-{retrival}{num_retrival}.txt"
global_descriptors = outputs / f"global-feats-{retrival}.h5"
query_descriptors = outputs_query / f"global-feats-{retrival}.h5"


In [ ]:
# Superpoint+SuperGlue
# feature_conf = extract_features.confs["superpoint_inloc"]
# matcher_conf = match_features.confs["superglue"]
# matcher_name = matcher_conf["output"]

# reconstruction = outputs / "sfm_superpoint+superglue"
# features = outputs / "feats-superpoint-n4096-r1600.h5"
# inlier_threshold = 32

# results = outputs / f"3259_hloc_query{query_set}_disk+{matcher_name}_{retrival}{num_retrival}.txt"  # the result file
### 0: 129/130 (+2 errors)  ,1: 138/143 ,2:125/127

In [ ]:
# Superpoint+SuperGlue_indoor
# feature_conf = extract_features.confs["superpoint_inloc"]
# matcher_conf = match_features.confs["superglue_indoor"]
# matcher_name = matcher_conf["output"]

# reconstruction = outputs / "sfm_superpoint+superglue_indoor"
# features = outputs / "feats-superpoint-n4096-r1600.h5"
# inlier_threshold = 32

# results = outputs / f"3259_hloc_query{query_set}_disk+{matcher_name}_{retrival}{num_retrival}.txt"  # the result file
### 0: 127 / 130  ,   1: 137/143   ,  2: 123/127

In [ ]:
# Superpoint+NN
# feature_conf = extract_features.confs["superpoint_inloc"]
# matcher_conf = match_features.confs["NN-superpoint"]
# matcher_name = matcher_conf["output"]

# reconstruction = outputs / "sfm_superpoint+NN"
# features = outputs / "feats-superpoint-n4096-r1600.h5"
# inlier_threshold = 32

# results = outputs / f"3259_hloc_query{query_set}_superpoint+{matcher_name}_{retrival}{num_retrival}.txt"  # the result file
### 0: 125/130  ,1: 134/143 ,2:119/127

In [ ]:
# Sift+NN
feature_conf = extract_features.confs["sift"]
matcher_conf = match_features.confs["NN-ratio"]
matcher_name = matcher_conf["output"]

reconstruction = outputs / "sfm_sift+NN_pruned"
features = outputs / "feats-sift.h5"
inlier_threshold = 10

results = outputs / f"3259_hloc_query{query_set}_sift+{matcher_name}_{retrival}{num_retrival}.txt"  # the result file
### 0:  126/130 ,1: 130/143   ,2: 120/127

In [ ]:
# DISK+LightGlue
# feature_conf = extract_features.confs["disk"]
# matcher_conf = match_features.confs["disk+lightglue"]
# matcher_name = matcher_conf["output"]

# reconstruction = outputs / "sfm_disk+lightglue"
# features = outputs / "feats-disk.h5"
# inlier_threshold = 40

# results = outputs / f"3259_hloc_query{query_set}_disk+{matcher_name}_{retrival}{num_retrival}.txt"  # the result file
### 0: 129/130 (+2 errors)   ,  1: 140/143 (+4 errors)   ,2:  124 / 127

In [ ]:
# DISK2+NN
# feature_conf = extract_features.confs["disk2"]
# matcher_conf = match_features.confs["NN-ratio"]
# matcher_name = matcher_conf["output"]

# reconstruction = outputs / "sfm_disk2+NN"
# features = outputs / "feats-disk2.h5"
# inlier_threshold = 10

# results = outputs / f"3259_hloc_query{query_set}_disk+{matcher_name}_{retrival}{num_retrival}.txt"  # the result file
### 0: 125/130  ,1: 130/143 ,2: 116/127

In [ ]:
# LOFTR
# matcher_conf = match_dense.confs['loftr']
# matcher_name = matcher_conf["output"]

# reconstruction = outputs / "sfm_loftr"
# features = outputs / f"feats_{matcher_name}.h5"
# inlier_threshold = 100

# query_matches = outputs_query / f"matches-loftr_{loc_pairs.stem}.h5"
# results = outputs / f"3259_hloc_loftr_{retrival}{num_retrival}.txt"
### 0: 128/130 (+2 errors)   ,  1: 137/143    ,2:  123/127

## Feature extraction Query

In [ ]:
# NOT FOR LOFTR
if matcher_conf["model"]["name"] != "loftr":
    features_query = extract_features.main(feature_conf, query_images, outputs_query)
    print(features_query)

In [ ]:
query_descriptors = extract_features.main(retrieval_conf, query_images, outputs_query)
print(query_descriptors)

In [ ]:
pairs_from_retrieval.main(
    query_descriptors, loc_pairs, num_matched=num_retrival, db_prefix=None,
    query_prefix=None, db_descriptors=global_descriptors
)

## Match the query images

In [ ]:
# NOT FOR LOFTR
if matcher_conf["model"]["name"] != "loftr":
    query_matches = Path( outputs_query, f'{features_query.stem}_{matcher_conf["output"]}_{loc_pairs.stem}.h5') ## Match OUTPUT location
    loc_matches = match_features.main(
        matcher_conf, loc_pairs, features_query, outputs_query,
        query_matches, features_ref=features
    )

In [ ]:
# LOFTR
if matcher_conf["model"]["name"] == "loftr":
    features_query, loc_matches = match_dense.main(
            matcher_conf,
            loc_pairs,
            dataset / "combined_with_query",
            outputs_query,
            max_kps=None,
            matches=query_matches,
            features_ref = features
        )

## Localize!

In [ ]:
# import h5py

# with h5py.File(features, 'r') as f:
#     # List all groups
#     data = f['1733767862507000064.jpg']['keypoints']
#     print(list(data)) 


In [ ]:
localize_sfm.main(
    reconstruction,
    dataset / f"query_list_{query_set}_with_intrinsics.txt",
    loc_pairs,
    features_query,
    loc_matches,
    results,
    covisibility_clustering=False,
    inlier_threshold = inlier_threshold,
)  # not required with SuperPoint+SuperGlue

## Visualizing the SfM model
We visualize some of the database images with their detected keypoints.

Color the keypoints by track length: red keypoints are observed many times, blue keypoints few.

In [ ]:
visualization.visualize_sfm_2d(reconstruction, images, n=1, color_by="track_length")

Color the keypoints by visibility: blue if sucessfully triangulated, red if never matched.

## Visualizing the localization
We parse the localization logs and for each query image plot matches and inliers with a few database images.

In [ ]:
selected = []
visualization.visualize_loc(
    results, query_images, reconstruction, db_image_dir= images, selected = selected, n=5, top_k_db=1, prefix=None, seed=4
)

In [ ]:
out_dict = read_images_text(results)
stamps, xyz, quat = colmap_dict_to_pose(out_dict)
df_pred = pd.DataFrame()
df_pred[["X", "Y", "Z"]] = xyz
df_pred["image_name"] = stamps
# stamps = stamps.reshape(-1, 1)
# final_out = np.hstack((xyz[:, 0:2], stamps))

In [ ]:
gt_file = dataset / f"query_list_{query_set}_gt.txt"
df_gt = pd.read_csv(gt_file, sep=' ', header=None, names=['image_name', 'X', 'Y', 'Z'])

In [ ]:
exclude_frames = []
# exclude_frames = ['1734377895414000128.jpg', '1734378021095000064.jpg']
df_plot = df_pred[~df_pred["image_name"].isin(exclude_frames)]
plt.plot(df_plot["X"], df_plot["Y"], color ="blue")
plt.plot(df_gt["X"], df_gt["Y"], color ="red")
plt.axis('equal')
print("plot")

In [ ]:
merged_df = pd.merge(df_pred, df_gt, on='image_name', how='inner', suffixes=('_pred', '_gt'))
del_x = merged_df['X_pred'] - merged_df['X_gt']
del_y = merged_df['Y_pred'] - merged_df['Y_gt']
error = del_x**2 + del_y**2
error = np.sqrt(error)

In [ ]:
error[error > 5]


In [ ]:
merged_df[error > 5]

In [ ]:
selected = list(merged_df[error > 5]["image_name"])
print(selected)

In [ ]:
error.describe()

In [ ]:
visualization.visualize_loc(
    results, query_images, reconstruction, db_image_dir= images, selected = selected, top_k_db=1, prefix=None, seed=4
)